<a href="https://colab.research.google.com/github/Hkd225/Financial-Fraud-Detection/blob/main/Financial_Fraud_Detection_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

path = kagglehub.dataset_download("sriharshaeedala/financial-fraud-detection-dataset")

100%|██████████| 178M/178M [00:01<00:00, 111MB/s]

Extracting files...


In [2]:
df = pd.read_csv(os.path.join(path, 'Synthetic_Financial_datasets_log.csv'))

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [4]:
from sklearn.preprocessing import OrdinalEncoder
X = df.drop("isFraud", axis=1)
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

cols_to_encode = ['type', 'nameOrig', 'nameDest']
X_train[cols_to_encode] = encoder.fit_transform(X_train[cols_to_encode])
X_test[cols_to_encode] = encoder.transform(X_test[cols_to_encode])


In [5]:
X_test.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFlaggedFraud
4051353,300,4.0,890577.21,-1.0,218.00,0.00,320401.0,0.00,890577.21,0
5746321,399,0.0,97734.24,-1.0,2096258.84,2193993.08,31243.0,320136.00,222401.76,0
6361797,718,3.0,5907.41,-1.0,315.00,0.00,-1.0,0.00,0.00,0
2247309,186,1.0,187696.30,-1.0,11057.00,0.00,120152.0,1798095.21,1985791.51,0
4692207,331,1.0,82646.52,-1.0,0.00,0.00,244077.0,1047805.87,1130452.39,0


In [6]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


In [7]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_scaled, y_train)

y_pred = model_rf.predict(X_test_scaled)

print(f"Akurasi Random Forest : {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

Akurasi Random Forest : 99.97%



In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model_rf.predict(X_test_scaled)

akurasi = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {akurasi * 100:.2f}%\n")
print("Laporan Klasifikasi:")
print(classification_report(y_test, y_pred))

Akurasi Model: 99.97%

Laporan Klasifikasi:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.98      0.77      0.86      1643

    accuracy                           1.00   1272524
   macro avg       0.99      0.89      0.93   1272524
weighted avg       1.00      1.00      1.00   1272524



In [9]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(penalty='l2', multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
logreg_model.fit(X_train_scaled, y_train)
y_pred_logreg = logreg_model.predict(X_test_scaled)

print("=== Hasil Logistic Regression (L2) ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred_logreg) * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(


=== Hasil Logistic Regression (L2) ===
Akurasi: 99.87%


In [10]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

print("=== Hasil Support Vector Machine (SVM) ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred_svm) * 100:.2f}%")

=== Hasil Support Vector Machine (SVM) ===
Akurasi: 99.89%


In [11]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print("=== Hasil XGBoost ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%")

=== Hasil XGBoost ===
Akurasi: 99.89%


In [12]:
print("\n--- LAPORAN KLASIFIKASI LENGKAP ---")

print("\n1. LOGISTIC REGRESSION (L2)")
print(classification_report(y_test, y_pred_logreg))

print("\n2. SUPPORT VECTOR MACHINE (SVM)")
print(classification_report(y_test, y_pred_svm))

print("\n3. XGBOOST")
print(classification_report(y_test, y_pred_xgb))

print("\n4. RANDOM FOREST")
print(classification_report(y_test, y_pred))


--- LAPORAN KLASIFIKASI LENGKAP ---

1. LOGISTIC REGRESSION (L2)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.00      0.00      0.00      1643

    accuracy                           1.00   1272524
   macro avg       0.50      0.50      0.50   1272524
weighted avg       1.00      1.00      1.00   1272524


2. SUPPORT VECTOR MACHINE (SVM)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      0.14      0.25      1643

    accuracy                           1.00   1272524
   macro avg       1.00      0.57      0.62   1272524
weighted avg       1.00      1.00      1.00   1272524


3. XGBOOST
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.58      0.52      0.55      1643

    accuracy                           1.00   1272524
   macro avg       0.79      0.76      0.77   1272524
weighted avg       1.00      

In [13]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

artifacts = {
    "encoder": encoder,
    "scaler": scaler,
    "random_forest_model": model_rf,
    "logistic_regression_model": logreg_model,
    "svm_model": svm_model,
    "xgboost_model": xgb_model,
    "feature_columns": X.columns.tolist(),
    "categorical_columns": cols_to_encode
}

joblib.dump(artifacts, "saved_models/fraud_detection_pipeline.pkl")

print("Semua model dan preprocessing berhasil disimpan!")

Semua model dan preprocessing berhasil disimpan!


In [14]:
import joblib

artifacts = joblib.load("saved_models/fraud_detection_pipeline.pkl")

encoder = artifacts["encoder"]
scaler = artifacts["scaler"]

model_rf = artifacts["random_forest_model"]
logreg_model = artifacts["logistic_regression_model"]
svm_model = artifacts["svm_model"]
xgb_model = artifacts["xgboost_model"]

feature_columns = artifacts["feature_columns"]
cat_cols = artifacts["categorical_columns"]

print("Model berhasil dimuat kembali!")

Model berhasil dimuat kembali!


In [20]:
import pandas as pd
import joblib

artifacts = joblib.load("saved_models/fraud_detection_pipeline.pkl")

encoder = artifacts["encoder"]
scaler = artifacts["scaler"]
model_rf = artifacts["random_forest_model"]
cat_cols = artifacts["categorical_columns"]
feature_cols = artifacts["feature_columns"]

new_data = pd.DataFrame([{
    "step":1,
    "type":"TRANSFER",
    "amount":1000,
    "nameOrig":"C123",
    "oldbalanceOrg":10000,
    "newbalanceOrig":9000,
    "nameDest":"C456",
    "oldbalanceDest":0,
    "newbalanceDest":1000,
    "isFlaggedFraud":0
}])

new_data[cat_cols] = encoder.transform(new_data[cat_cols])

new_data = new_data[feature_cols]

new_data_scaled = scaler.transform(new_data)

new_data_scaled = pd.DataFrame(new_data_scaled, columns=feature_cols)

pred = model_rf.predict(new_data_scaled)

print("Prediksi Fraud:", pred)

Prediksi Fraud: [0]
